<a href="https://colab.research.google.com/github/Adhisha11/Word_Sense_Disambiguation/blob/main/Word_Sense_Disambiguation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import nltk
import pandas as pd
import re

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

print("Libraries loaded successfully!")


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Libraries loaded successfully!


In [2]:

word = "bank"

for i, syn in enumerate(wn.synsets(word), start=1):
    print(f"\nSense {i}: {syn.name()}")
    print("Definition:", syn.definition())
    print("Synonyms:", syn.lemma_names())

    hypernyms = syn.hypernyms()
    if hypernyms:
        print("Hypernyms:",
              [h.name() for h in hypernyms])



Sense 1: bank.n.01
Definition: sloping land (especially the slope beside a body of water)
Synonyms: ['bank']
Hypernyms: ['slope.n.01']

Sense 2: depository_financial_institution.n.01
Definition: a financial institution that accepts deposits and channels the money into lending activities
Synonyms: ['depository_financial_institution', 'bank', 'banking_concern', 'banking_company']
Hypernyms: ['financial_institution.n.01']

Sense 3: bank.n.03
Definition: a long ridge or pile
Synonyms: ['bank']
Hypernyms: ['ridge.n.01']

Sense 4: bank.n.04
Definition: an arrangement of similar objects in a row or in tiers
Synonyms: ['bank']
Hypernyms: ['array.n.01']

Sense 5: bank.n.05
Definition: a supply or stock held in reserve for future use (especially in emergencies)
Synonyms: ['bank']
Hypernyms: ['reserve.n.02']

Sense 6: bank.n.06
Definition: the funds held by a gambling house or the dealer in some gambling games
Synonyms: ['bank']
Hypernyms: ['funds.n.01']

Sense 7: bank.n.07
Definition: a slope i

In [3]:

samples = [
    {
        "sentence": "I deposited money in the bank.",
        "word": "bank"
    },
    {
        "sentence": "The fisherman sat on the bank of the river.",
        "word": "bank"
    },
    {
        "sentence": "The bat flew out of the cave at night.",
        "word": "bat"
    },
    {
        "sentence": "The player hit the ball with a bat.",
        "word": "bat"
    },
    {
        "sentence": "Please turn on the light in the room.",
        "word": "light"
    },
    {
        "sentence": "This bag is very light to carry.",
        "word": "light"
    }
]

df = pd.DataFrame(samples)
df


,sentence,word
0,I deposited money in the bank.,bank
1,The fisherman sat on the bank of the river.,bank
2,The bat flew out of the cave at night.,bat
3,The player hit the ball with a bat.,bat
4,Please turn on the light in the room.,light
5,This bag is very light to carry.,light


In [4]:

def disambiguate(sentence, word):
    tokens = re.findall(r"[a-zA-Z]+", sentence.lower())

    sense = lesk(tokens, word.lower())

    if sense is None:
        return {
            "predicted_sense": "Not found",
            "definition": "No WordNet sense found"
        }

    return {
        "predicted_sense": sense.name(),
        "definition": sense.definition()
    }


results = []

for item in samples:
    prediction = disambiguate(
        item["sentence"],
        item["word"]
    )

    results.append({
        "Sentence": item["sentence"],
        "Target Word": item["word"],
        "Predicted Sense": prediction["predicted_sense"],
        "Definition": prediction["definition"]
    })

results_df = pd.DataFrame(results)
display(results_df)


,Sentence,Target Word,Predicted Sense,Definition
0,I deposited money in the bank.,bank,savings_bank.n.02,a container (usually with a slot in the top) f...
1,The fisherman sat on the bank of the river.,bank,bank.v.07,cover with ashes so to control the rate of bur...
2,The bat flew out of the cave at night.,bat,bat.v.03,have a turn at bat
3,The player hit the ball with a bat.,bat,bat.v.01,"strike with, or as if with a baseball bat"
4,Please turn on the light in the room.,light,light.n.07,the visual effect of illumination on objects o...
5,This bag is very light to carry.,light,light.s.22,designed for ease of movement or to carry litt...


In [5]:

words = ["bank", "bat", "light"]
inventory = []

for word in words:
    for sense in wn.synsets(word):
        inventory.append({
            "Word": word,
            "Sense ID": sense.name(),
            "Part of Speech": sense.pos(),
            "Definition": sense.definition(),
            "Synonyms": ", ".join(sense.lemma_names()),
            "Examples": "; ".join(sense.examples())
        })

inventory_df = pd.DataFrame(inventory)

display(inventory_df.head(15))
print("Total senses found:", len(inventory_df))


,Word,Sense ID,Part of Speech,Definition,Synonyms,Examples
0,bank,bank.n.01,n,sloping land (especially the slope beside a bo...,bank,they pulled the canoe up on the bank; he sat o...
1,bank,depository_financial_institution.n.01,n,a financial institution that accepts deposits ...,"depository_financial_institution, bank, bankin...",he cashed a check at the bank; that bank holds...
2,bank,bank.n.03,n,a long ridge or pile,bank,a huge bank of earth
3,bank,bank.n.04,n,an arrangement of similar objects in a row or ...,bank,he operated a bank of switches
4,bank,bank.n.05,n,a supply or stock held in reserve for future u...,bank,
5,bank,bank.n.06,n,the funds held by a gambling house or the deal...,bank,he tried to break the bank at Monte Carlo
6,bank,bank.n.07,n,a slope in the turn of a road or track; the ou...,"bank, cant, camber",
7,bank,savings_bank.n.02,n,a container (usually with a slot in the top) f...,"savings_bank, coin_bank, money_box, bank",the coin bank was empty
8,bank,bank.n.09,n,a building in which the business of banking tr...,"bank, bank_building",the bank is on the corner of Nassau and Wither...
9,bank,bank.n.10,n,a flight maneuver; aircraft tips laterally abo...,bank,the plane went into a steep bank


Total senses found: 75


In [6]:

inventory_df.to_csv("sense_inventory.csv", index=False)
results_df.to_csv("wsd_results.csv", index=False)

print("Files created successfully!")


Files created successfully!
